<a href="https://colab.research.google.com/github/mofermino/condensed-subject-matters/blob/main/triangle_osc.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
# This script generates an interactive 3D HTML animation (no internet needed) that visualizes
# the in‑plane vibrational modes (A1' breathing and the E' doublet) of an equilateral
# triatomic molecule (3 equal masses, springs on each edge). It provides checkboxes to toggle
# each mode on/off live. Plotly JS is inlined, so the file is fully self-contained.
#
# Output: /content/data/triatomic_modes.html

import json
import numpy as np
import sympy as sp
import plotly.graph_objects as go
import plotly.io as pio

# Geometry: equilateral triangle of side a = 1 (arbitrary units)
a = 1.0
r1 = np.array([ a, 0.0, 0.0])
r2 = np.array([-a/2,  np.sqrt(3)/2*a, 0.0])
r3 = np.array([-a/2, -np.sqrt(3)/2*a, 0.0])
R0 = np.stack([r1, r2, r3], axis=0)  # (3,3)

# Build harmonic dynamical matrix for in-plane motion (x,y for each atom), central springs K (we take K/m=1 for geometry of eigenvectors)
# Bonds unit vectors (in plane)
def unit(v):
    v = v / np.linalg.norm(v)
    return v

n12 = unit(r2 - r1)[:2]
n23 = unit(r3 - r2)[:2]
n31 = unit(r1 - r3)[:2]
dyads = [np.outer(n12, n12), np.outer(n23, n23), np.outer(n31, n31)]

D = np.zeros((6,6), dtype=float)

def put_block(i,j,B):
    D[2*i:2*i+2, 2*j:2*j+2] += B

def add_bond(i,j,M):
    put_block(i,i,M); put_block(j,j,M)
    put_block(i,j,-M); put_block(j,i,-M)

add_bond(0,1,dyads[0])
add_bond(1,2,dyads[1])
add_bond(2,0,dyads[2])

# Eigen-decomposition
w, V = np.linalg.eigh(D)  # w are eigenvalues of D (proportional to m*omega^2 when K/m=1)
# Sort by eigenvalue
idx = np.argsort(w)
w = w[idx]; V = V[:,idx]

# Identify modes: three zeros (translations + in-plane rotation), then E' doublet (3/2), then A1' (3)
# Numerical tolerances:
zero_tol = 1e-8
# Collect eigenvectors for nonzero modes
nonzero_inds = np.where(w > zero_tol)[0]
# Expect two at ~1.5, one at ~3.0
# Make robust grouping:
vals = w[nonzero_inds]
# Group by closeness
groups = {}
for i, lam in zip(nonzero_inds, vals):
    key = "E" if abs(lam - 1.5) < 1e-4 else ("A1" if abs(lam - 3.0) < 1e-4 else f"lam_{lam:.3f}")
    groups.setdefault(key, []).append(i)

E_inds = groups.get("E", [])[:2]
A1_inds = groups.get("A1", [])[:1]

# Build full 3D eigenvectors (extend 2D in-plane to 3D by adding zeros in z for each atom)
def vec6_to_vec9(v6):
    # v6 = [x1,y1,x2,y2,x3,y3]
    v9 = np.zeros((3,3), dtype=float)
    v9[0,:2] = v6[0:2]
    v9[1,:2] = v6[2:4]
    v9[2,:2] = v6[4:6]
    return v9.reshape(-1)  # 9

# Normalize modes to convenient visualization amplitude (max component = 1)
def normalize(v):
    vmax = np.max(np.abs(v))
    if vmax < 1e-12:
        return v
    return v / vmax

# Extract and normalize
E_modes = []
for i in E_inds:
    v6 = V[:, i]
    v9 = vec6_to_vec9(v6)
    E_modes.append(normalize(v9))
A1_mode = None
if A1_inds:
    v6 = V[:, A1_inds[0]]
    v9 = vec6_to_vec9(v6)
    # Enforce breathing orientation: align with radial (optional)
    A1_mode = normalize(v9)

# Frequencies with K/m = 1 for convenience:
# D eigvals equal m*omega^2 when K/m=1, so omega = sqrt(w)
omega_E = float(np.sqrt(1.5))
omega_A1 = float(np.sqrt(3.0))

# Build Plotly figure: masses (spheres as markers) + springs (lines)
# Mass positions at t=0:
X = R0[:,0]; Y = R0[:,1]; Z = R0[:,2]

masses = go.Scatter3d(
    x=X, y=Y, z=Z,
    mode="markers",
    marker=dict(size=6),
    name="masses"
)

# Springs as line segments with None separators
def spring_lines(coords):
    # coords shape (3,3)
    lines = []
    edges = [(0,1),(1,2),(2,0)]
    for i,j in edges:
        lines.append(coords[i])
        lines.append(coords[j])
        lines.append([None, None, None])
    lines = np.array(lines, dtype=object)
    xs = [p[0] if p is not None else None for p in lines]
    ys = [p[1] if p is not None else None for p in lines]
    zs = [p[2] if p is not None else None for p in lines]
    return xs, ys, zs

xs, ys, zs = spring_lines(R0)
springs = go.Scatter3d(
    x=xs, y=ys, z=zs,
    mode="lines",
    line=dict(width=4),
    name="springs"
)

fig = go.Figure(data=[masses, springs])
fig.update_layout(
    title="Triatomic Molecule — In‑Plane Vibrational Modes (A1′ & E′)",
    scene=dict(
        aspectmode='data',
        xaxis=dict(visible=False),
        yaxis=dict(visible=False),
        zaxis=dict(visible=False),
    ),
    showlegend=False,
    margin=dict(l=0, r=0, t=30, b=0),
)

# Export base HTML with inlined Plotly
html_base = pio.to_html(fig, include_plotlyjs=True, full_html=True, auto_play=False)

# Insert custom controls and animation JS into the HTML
# Prepare mode data for JS: R0 (3x3), modes E1, E2, A1 (each 9 entries flattened by atom then xyz)
def arr(a):
    return a.tolist()

R0_js = R0.tolist()
E1_js = E_modes[0].reshape(3,3).tolist() if len(E_modes)>=1 else [[0,0,0]]*3
E2_js = E_modes[1].reshape(3,3).tolist() if len(E_modes)>=2 else [[0,0,0]]*3
A1_js = A1_mode.reshape(3,3).tolist() if A1_mode is not None else [[0,0,0]]*3

custom_controls = f"""
<div style="position:relative; padding:8px 12px; font-family:system-ui, -apple-system, Segoe UI, Roboto, sans-serif;">
  <strong>Mode toggles</strong> — A<sub>1</sub>′ (breathing) and E′ doublet
  <div style="display:flex; gap:18px; padding-top:6px; align-items:center; flex-wrap:wrap;">
    <label><input type="checkbox" id="toggleA1" checked> A<sub>1</sub>′ (ω = √3)</label>
    <label><input type="checkbox" id="toggleE1" checked> E′<sub>1</sub> (ω = √(3/2))</label>
    <label><input type="checkbox" id="toggleE2" checked> E′<sub>2</sub> (ω = √(3/2))</label>
    <label style="margin-left:12px;">Amplitude:
      <input type="range" id="amp" min="0" max="0.6" step="0.01" value="0.25">
      <span id="ampVal">0.25</span>
    </label>
    <label>Speed:
      <input type="range" id="spd" min="0.25" max="3" step="0.05" value="1.0">
      <span id="spdVal">1.00×</span>
    </label>
    <button id="pauseBtn">Pause</button>
  </div>
</div>
"""

animation_js = f"""
<script>
(function() {{
  const fig = document.querySelector('div.js-plotly-plot');
  // Base positions and modes (arrays of 3 atoms x 3 coords)
  const R0 = {json.dumps(R0_js)};
  const E1 = {json.dumps(E1_js)};
  const E2 = {json.dumps(E2_js)};
  const A1 = {json.dumps(A1_js)};

  const omegaE = Math.sqrt(1.5);
  const omegaA = Math.sqrt(3.0);

  const toggleA1 = document.getElementById('toggleA1');
  const toggleE1 = document.getElementById('toggleE1');
  const toggleE2 = document.getElementById('toggleE2');
  const amp = document.getElementById('amp');
  const ampVal = document.getElementById('ampVal');
  const spd = document.getElementById('spd');
  const spdVal = document.getElementById('spdVal');
  const pauseBtn = document.getElementById('pauseBtn');

  amp.addEventListener('input', () => ampVal.textContent = parseFloat(amp.value).toFixed(2));
  spd.addEventListener('input', () => spdVal.textContent = parseFloat(spd.value).toFixed(2) + '×');

  let running = true;
  pauseBtn.addEventListener('click', () => {{
    running = !running;
    pauseBtn.textContent = running ? 'Pause' : 'Resume';
  }});

  let t0 = performance.now()/1000;
  function step() {{
    if (!running) {{ requestAnimationFrame(step); return; }}
    const t = (performance.now()/1000 - t0) * parseFloat(spd.value);

    const A = parseFloat(amp.value);
    const aA = toggleA1.checked ? A*Math.cos(omegaA * t) : 0.0;
    const aE1 = toggleE1.checked ? A*Math.cos(omegaE * t) : 0.0;
    const aE2 = toggleE2.checked ? A*Math.sin(omegaE * t) : 0.0; // phase shift for variety

    // New positions: R(t) = R0 + aA*A1 + aE1*E1 + aE2*E2
    const Rt = [[0,0,0],[0,0,0],[0,0,0]];
    for (let i=0;i<3;i++) {{
      for (let c=0;c<3;c++) {{
        Rt[i][c] = R0[i][c] + aA*A1[i][c] + aE1*E1[i][c] + aE2*E2[i][c];
      }}
    }}

    // Update masses (trace 0) and spring lines (trace 1)
    const xs = [Rt[0][0], Rt[1][0], Rt[2][0]];
    const ys = [Rt[0][1], Rt[1][1], Rt[2][1]];
    const zs = [Rt[0][2], Rt[1][2], Rt[2][2]];

    const lineX = [Rt[0][0], Rt[1][0], null, Rt[1][0], Rt[2][0], null, Rt[2][0], Rt[0][0], null];
    const lineY = [Rt[0][1], Rt[1][1], null, Rt[1][1], Rt[2][1], null, Rt[2][1], Rt[0][1], null];
    const lineZ = [Rt[0][2], Rt[1][2], null, Rt[1][2], Rt[2][2], null, Rt[2][2], Rt[0][2], null];

    Plotly.restyle(fig, {{x: [xs], y: [ys], z: [zs]}}, [0]); // masses
    Plotly.restyle(fig, {{x: [lineX], y: [lineY], z: [lineZ]}}, [1]); // springs

    requestAnimationFrame(step);
  }}
  step();
}})();
</script>
"""

# Inject controls and JS into the HTML
# Insert controls above the plot div
insert_at = html_base.find('<div id="')
html_out = html_base[:insert_at] + custom_controls + html_base[insert_at:]
# Append animation JS before closing body
html_out = html_out.replace('</body>', animation_js + '\n</body>')

out_path = '/content/triatomic_modes.html'
with open(out_path, 'w', encoding='utf-8') as f:
    f.write(html_out)

out_path


'/content/triatomic_modes.html'